# GCM BA - 3. Apply F1 to EC-Earth3 historical and to the four GCM runs

The **same** F1 (notebook 2: fixed WDF + QDM, EC-Earth3 grid) is applied unchanged to:
EC-Earth3 historical (for the verification), 2025 original, 2025 hosing, 2050 original, 2050 hosing.
Result: **"ERA5-like" GCM runs on the EC-Earth3 grid (0.70 deg)**, inside the mask. The 2025 original one is the
reference for the bias adjustment of the RCM. `rsds` is adjusted for the runs where it exists.

One file per run: `3_F1_application/F1_<period>/<run>_F1.nc`.

```
GCM_downscaling/GCM_BA/                 (EOS, next to Input_data/)
├── 1_regrid_and_mask/   <dataset>_<var>.nc : ERA5 on the EC-Earth3 grid, EC-Earth3 historical, 4 GCM runs, masked
├── 2_F1_calculation/    F1_<period>/       : WDF rule + QDM parameters
└── 3_F1_application/    F1_<period>/<run>_F1.nc : historical + 4 GCM runs, ERA5-like on the EC-Earth3 grid
```

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS_CORE = cfg["variables"]                                   # pr, tasmax, tasmin: required
VARS_ALL = VARS_CORE + cfg.get("extra_variables", [])          # + rsds where it exists
GCM_RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]

CAL_PERIOD = ("1981", "2014")      # years of ERA5 / EC-Earth3 historical used for F1 (EC-Earth3 historical ends in 2014)
RSDS_FIRST_YEAR = "2005"           # ERA5 radiation starts in 2005: shorter period for rsds

def period_of(v):
    """Calibration period of a variable: CAL_PERIOD, starting in 2005 for rsds."""
    return (max(CAL_PERIOD[0], RSDS_FIRST_YEAR), CAL_PERIOD[1]) if v == "rsds" else CAL_PERIOD

IN = bu.gcm_ba_dir(cfg, "1_regrid_and_mask")                                       # prepared inputs (EOS)
F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}"
CHUNKS_SPACE = {"lat": 32, "lon": 64}
OVERWRITE = False
F1_DIR = bu.gcm_ba_dir(cfg, "2_F1_calculation", F1_TAG)
OUT = bu.gcm_ba_dir(cfg, "3_F1_application", F1_TAG)
assert (F1_DIR / "qdm_pr.nc").exists(), "F1 not calibrated: run notebook 2 first."
RUNS = ["gcm_historical"] + GCM_RUNS
print("F1:", F1_DIR, "| outputs:", OUT)

## 1. Application

In [ ]:
for name in RUNS:
    out_path = OUT / f"{name}_F1.nc"
    if out_path.exists() and not OVERWRITE:
        print(f"[already done] {name}"); continue
    vs = [v for v in VARS_ALL if (IN / f"{name}_{v}.nc").exists()]
    if not all(v in vs for v in VARS_CORE):
        print(f"[missing] {name}: prepared inputs missing (notebook 1)"); continue
    print(f"-> {name}: {', '.join(vs)}")
    sim = {v: bu.read_processed(IN / f"{name}_{v}.nc", v) for v in vs}
    adj = bu.apply_F(sim, F1_DIR, chunks_space=CHUNKS_SPACE)
    bu.save_ds(xr.Dataset(adj), out_path)

## 2. Quick checks (first 3 years, inside the mask): pr >= 0, tasmin < tasmax, no new NaN

In [ ]:
rows = []
for name in RUNS:
    f = OUT / f"{name}_F1.nc"
    if not f.exists():
        continue
    ds = xr.open_dataset(f, chunks={"time": 365}).isel(time=slice(0, 365 * 3))
    raw = bu.read_processed(IN / f"{name}_pr.nc", "pr").isel(time=slice(0, 365 * 3))
    inside = raw.notnull()
    rows.append({"run": name, "variables": ", ".join(ds.data_vars),
                 "pr min": float(ds["pr"].min()),
                 "frac tasmin >= tasmax (should be 0)": float(((ds["tasmin"] >= ds["tasmax"]) & inside).sum() / inside.sum()),
                 "NaN inside the mask (should be 0)": float((ds["pr"].isnull() & inside).sum() / inside.sum())})
display(pd.DataFrame(rows).set_index("run"))

## 3. Amplification of precipitation by F1 (first 3 years, inside the mask)

Multiplicative QDM can produce very large correction factors where the model is much drier than the reference at a given
quantile. Ratio F1 / raw on wet days (raw >= 1 mm/day): maximum, 99.9th percentile, share of days amplified more than
x5, and the largest adjusted values. Large factors on many days, or adjusted values far above the raw maxima, must be
understood before using the adjusted runs (if needed: cap the factor, or more quantiles / longer calibration).

In [ ]:
rows = []
for name in RUNS:
    f = OUT / f"{name}_F1.nc"
    if not f.exists():
        continue
    adj = xr.open_dataset(f, chunks={"time": 365})["pr"].isel(time=slice(0, 365 * 3))
    raw = bu.read_processed(IN / f"{name}_pr.nc", "pr").isel(time=slice(0, 365 * 3))
    wet = raw >= 1.0
    ratio = (adj / raw).where(wet).values.ravel()
    ratio = ratio[np.isfinite(ratio)]
    rows.append({"run": name, "wet days x pixels": ratio.size, "max factor": round(float(ratio.max()), 2),
                 "p99.9 factor": round(float(np.percentile(ratio, 99.9)), 2),
                 "frac factor > 5 (%)": round(100 * float((ratio > 5).mean()), 4),
                 "raw max (mm/day)": round(float(raw.max()), 1), "F1 max (mm/day)": round(float(adj.max()), 1)})
display(pd.DataFrame(rows).set_index("run"))